In [ ]:
# Inteligencia de Negocios

## Componente Práctico Semana 2

## Grupo # 3

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ
### PABLO RAMIRO VALLEJO ZÚÑIGA

### Importación de dependencias

Se importan las librerías necesarias para:
- **pandas**: manipulación y análisis de DataFrames
- **sqlalchemy + psycopg2**: conexión ORM a PostgreSQL
- **json**: lectura del archivo de vulnerabilidades
- **dotenv**: carga de credenciales desde `.env` (buena práctica de seguridad)


In [1]:
import pandas as pd
import json
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
print("Dependencias cargadas correctamente")

Dependencias cargadas correctamente


### Manejo de variables de entorno

Las credenciales de la base de datos se leen del archivo `.env` para evitar exponer datos sensibles en el código fuente.
Esta práctica es recomendable para no colocar las credenciales directamente en el código fuente.
Se emplearon las variables DB_USER, DB_PASSWORD, DB_NAME, DB_HOST Y DB_PORT.
Como norma general no se deben exponer públicamente ningún tipo de credenciales, por ende no deben constan en repositorios públicos o directamente en el código fuente.
También debemos tomar en cuenta que, al publicar los proyectos en repositorios como GitHub, los archivos .env sean excluídos de la carga mediante la creación de los archivos .gitignore.


In [2]:
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")

print(f" Conectando a: {DB_HOST}:{DB_PORT}/{DB_NAME} como usuario '{DB_USER}'")

 Conectando a: localhost:5432/dbmcib12b como usuario 'admin'


# Desarrollo
## 1. Exploración inicial de las fuentes de datos

De acuerdo al análisis realizado, las fuentes de datos seleccionadas en el Componente Práctico de la semana 1 no poseen campos relacionales que nos permitan vincularlas en una sola tabla compuesta, por lo que hemos decidido realizar una nueva selección de fuentes de datos, la cual está vinculada al análisis de ciberseguridad, y contiene información sobre los equipos de la empresa, logs de acceso y registros de tráfico de red.

## 1.1. Base de datos postgres

Empleamos la base de datos llamada assets que contiene información sobre los dispositivos de la empresa.

### Creación de data frame desde tabla de datos postgres assets

In [12]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df_assets = pd.read_sql("SELECT * FROM assets", engine)


In [13]:
df_assets

,device_ip,device_name,department,criticality_level
0,192.168.140.208,SRV-ELECTION-91,Finanzas,Media
1,10.52.122.63,SRV-REQUIRE-13,IT-Operaciones,Baja
2,192.168.44.131,SRV-PERFORMANCE-41,Tecnología,Baja
3,172.18.255.141,SRV-DISCOVER-27,Finanzas,Media
4,10.13.150.31,SRV-OPTION-96,Legal,Alta
...,...,...,...,...
1045,172.26.207.139,SRV-LEADER-67,RRHH,Media
1046,192.168.175.122,SRV-WEEK-97,Legal,Media
1047,172.22.7.150,SRV-COMMUNITY-54,Tecnología,Alta
1048,172.23.45.163,SRV-SHORT-80,Legal,Baja


Estructura General

In [14]:
df_assets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   device_ip          1050 non-null   str  
 1   device_name        1050 non-null   str  
 2   department         1050 non-null   str  
 3   criticality_level  1050 non-null   str  
dtypes: str(4)
memory usage: 32.9 KB


In [15]:
df_assets.isnull().any()

device_ip            False
device_name          False
department           False
criticality_level    False
dtype: bool

In [16]:
df_assets.drop_duplicates()

,device_ip,device_name,department,criticality_level
0,192.168.140.208,SRV-ELECTION-91,Finanzas,Media
1,10.52.122.63,SRV-REQUIRE-13,IT-Operaciones,Baja
2,192.168.44.131,SRV-PERFORMANCE-41,Tecnología,Baja
3,172.18.255.141,SRV-DISCOVER-27,Finanzas,Media
4,10.13.150.31,SRV-OPTION-96,Legal,Alta
...,...,...,...,...
1045,172.26.207.139,SRV-LEADER-67,RRHH,Media
1046,192.168.175.122,SRV-WEEK-97,Legal,Media
1047,172.22.7.150,SRV-COMMUNITY-54,Tecnología,Alta
1048,172.23.45.163,SRV-SHORT-80,Legal,Baja


El data frame df_assets contiene 1050 registros clasificados en 4 columnas, que nos dan datos sobre la dirección IP del equipo, el nombre del mismo, el departamento al que pertenece y el nivel de criticidad de dicho equipo.

Todas las columnas en este data frame son de tipo string y no poseen valores nulos ni duplicados que limpiar.

De este data frame nos parecen de particular importancia las columnas device_ip y department para ser relacionadas con las otras fuentes de datos.

El segundo data frame contiene logs del tráfico de red. No tenemos valores nulos por limpiar.

In [19]:
df_network = pd.read_csv('data/network_traffic_large.csv')

df_network

,Source_IP,Destination_IP,Timestamp,Protocol,Total_Packets,Label
0,172.22.78.28,219.187.29.255,13/06/2026 08:00:32,ICMP,11028,Brute Force
1,172.31.118.104,200.38.66.153,2026-06-13 08:00:56,TCP,178,Benign
2,10.122.34.144,10.211.156.60,2026-06-13 08:01:34,TCP,41961,Port Scan
3,10.94.203.245,171.5.237.152,2026-06-13 08:01:58,TCP,39784,DoS
4,192.168.78.253,204.98.57.12,2026-06-13 08:02:40,ICMP,3699,Botnet
...,...,...,...,...,...,...
1195,172.20.80.182,56.64.19.255,2026-06-13 18:48:03,TCP,409,Benign
1196,10.216.139.45,131.239.83.32,2026-06-13 18:49:01,TCP,213,Benign
1197,192.168.41.70,10.146.251.194,2026-06-13 18:49:38,ICMP,14573,DoS
1198,172.19.214.84,172.19.230.46,2026-06-13 18:50:17,TCP,183,Benign


In [21]:
df_auth = pd.read_json('data/auth_logs_large.json')

df_auth

,event_id,host_ip,user_id,status,time
0,10000,10.53.158.133,shermandonna,Success,2026-06-13T07:55:18
1,10001,172.29.167.220,yschmidt,Failed,2026-06-13T07:55:37
2,10002,10.71.212.242,rgonzalez,Success,2026-06-13T07:55:59
3,10003,172.23.233.99,kevin46,Failed,2026-06-13T07:56:06
4,10004,172.17.198.40,veronica90,Failed,2026-06-13T07:56:24
...,...,...,...,...,...
1295,11295,10.39.221.155,ynichols,Success,2026-06-13T16:35:46
1296,11296,10.66.250.203,tammykim,Success,2026-06-13T16:36:09
1297,11297,172.18.54.140,jmorrow,Success,2026-06-13T16:36:34
1298,11298,192.168.47.213,christiancontreras,Success,2026-06-13T16:37:17
